# CausalFoundations: Folktables Exploration & SCM Prototyping

Notebook inicial para explorar os microdados do **Folktables (ACS PUMS)** e prototipar a injeção de Modelos Causais Estruturais (SCMs).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from causal_foundations import (
    load_acs_data,
    generate_synthetic_target,
    inject_spurious_shortcut,
    get_tabpfn_classifier,
    get_baseline_classifiers,
)

## 1. Carregar Dados do Censo via Folktables

In [ ]:
# Exemplo: Carregar amostra de 1000 registros da Califórnia (2018) para a tarefa ACSIncome
X, y, group = load_acs_data(state="CA", year="2018", task_name="ACSIncome", sample_size=1000)

print(f"Shape de X: {X.shape}")
X.head()

## 2. Inspecionar Covariáveis Reais

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(X["AGEP"], bins=20, ax=axes[0], kde=True, color="steelblue")
axes[0].set_title("Idade (AGEP)")
sns.histplot(X["WKHP"], bins=20, ax=axes[1], kde=True, color="teal")
axes[1].set_title("Horas de trabalho/semana (WKHP)")
plt.tight_layout()
plt.show()

## 3. Injeção de SCM e Criação de Atalho Espúrio

Aqui injetamos um alvo sintético $Y = f(Pa(Y)) + \epsilon$ com pais causais controlados (`AGEP`, `SCHL`, `WKHP`) e um atalho espúrio $S$.

In [ ]:
# Gerar alvo sintético a partir dos pais causais escolhidos
causal_parents = ["AGEP", "SCHL", "WKHP"]
causal_weights = [0.3, 0.5, 0.2]

y_synthetic, prob = generate_synthetic_target(
    X, causal_parents=causal_parents, weights=causal_weights
)

# Injetar um atalho fortemente correlacionado com o alvo
spurious_feature = inject_spurious_shortcut(y_synthetic, correlation=0.85)
X_experiment = X.copy()
X_experiment["SPURIOUS_SIGNAL"] = spurious_feature

print(f"Correlação de Pearson com Y: {np.corrcoef(spurious_feature, y_synthetic)[0, 1]:.3f}")
X_experiment.head()

## 4. Próximos Passos para o Grupo

- Treinar TabPFN (`get_tabpfn_classifier()`) e baselines (`get_baseline_classifiers()`).
- Quebrar a correlação espúria em teste (ex: avaliar em outro estado como TX ou com ruído).
- Extrair valores SHAP para verificar se os modelos confiam nos pais causais ou no sinal espúrio.